# MOFA-Flex evaluation

**MOFA-Flex** (`factor_prior="Normal"`, `weight_prior="SpikeSlab"`) as an additional method
alongside the 25 conventional methods and MOFA+, on the same three clinical matrices
(`dlbcl_rna`, `dlbcl_protein`, `crc_protein`) and the same masking design.

Just like other 25 conventional methods in `benchmark_imputation_clinical.ipynb` and MOFA+ in  `evaluate_mofaplus_multiomic_clinical.ipynb`, MOFA-Flex's rows land in the *same*
`pilot_checkpoint.csv` / `benchmark_checkpoint_<matrix>.csv` where they already write to - it shows up as one more `Method` in the same tables, not a separate
analysis to cross-reference.

**Prior-choice alignment with MOFA+**:
* **Factor prior ($\mathbf{Z}$):** `factor_prior="Normal"` - standard $\mathcal{N}(\mathbf{0}, \mathbf{I})$, matching MOFA+'s default.
* **Loading prior ($\mathbf{W}$):** `weight_prior="SpikeSlab"` - entry-wise Bernoulli sparsity + per-factor ARD, equivalent to MOFA+'s `spikeslab_weights=True, ard_weights=True`.
* Missing entries are handled natively during variational inference (evaluated only over observed positions), same as MOFA+ - no pre-imputation.
* Where MOFA-Flex differs from MOFA+ is *inference*: stochastic variational inference (SVI, Pyro-backed, GPU) instead of MOFA+'s closed-form coordinate-ascent - so results are not expected to be identical even under matched priors, and SVI introduces real seed-to-seed variance that coordinate-ascent doesn't have.

### Requirements

`mofaflex` + a CUDA-capable `torch` build (`pip install mofaflex`, Python >= 3.11; see
https://github.com/bioFAM/mofaflex). This notebook will not run in the same environment as the
25-method benchmark unless that environment also has `mofaflex`/`torch` installed. 

In [1]:
# --- Cell 1: cap CPU thread usage, imports, path resolution, mofaflex/torch guard ---
import os
import time
import warnings
from pathlib import Path


def _cluster_cpu_budget(default=4):
    """Same priority order as benchmark_imputation_clinical.ipynb/evaluate_mofaplus_
    multiomic_clinical.ipynb's own Cell 1: an explicit IMPUTATION_CPU_BUDGET override
    beats the Slurm allocation, which beats this process's own CPU affinity, which
    beats a conservative default."""
    override = os.environ.get("IMPUTATION_CPU_BUDGET")
    if override:
        try:
            return max(1, int(override)), "IMPUTATION_CPU_BUDGET override"
        except ValueError:
            print(f"IMPUTATION_CPU_BUDGET={override!r} isn't a valid integer - ignoring it.")

    for var in ("SLURM_CPUS_PER_TASK", "SLURM_CPUS_ON_NODE"):
        val = os.environ.get(var)
        if val:
            try:
                return max(1, int(val)), var
            except ValueError:
                pass
    val = os.environ.get("SLURM_JOB_CPUS_PER_NODE")
    if val:
        import re
        m = re.match(r"(\d+)", val)
        if m:
            return max(1, int(m.group(1))), "SLURM_JOB_CPUS_PER_NODE"

    try:
        return max(1, len(os.sched_getaffinity(0))), "CPU affinity (no Slurm allocation, no override detected)"
    except AttributeError:
        return max(1, os.cpu_count() or default), "os.cpu_count() (no Slurm allocation, no override detected)"


N_JOBS, _cpu_budget_source = _cluster_cpu_budget()
for _var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_var] = str(N_JOBS)
print(f"CPU budget for this session: {N_JOBS} threads (source: {_cpu_budget_source})")

import numpy as np
import pandas as pd
import mudata as mu

mu.set_options(pull_on_update=False)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

RNG_SEED = 0  # keep in sync with benchmark_imputation_clinical.ipynb/py /
              # evaluate_mofaplus_multiomic_clinical.ipynb / evaluate_imputation_methods_clinical.ipynb

# --- mofaflex/torch guard -- same pattern as MOFAPLUS_AVAILABLE in
# evaluate_mofaplus_multiomic_clinical.ipynb's Cell 1.
RUN_MOFAFLEX = True
MOFAFLEX_AVAILABLE = False
DEVICE = "cpu"
if RUN_MOFAFLEX:
    try:
        import torch
        import mofaflex as mfl
        MOFAFLEX_AVAILABLE = True
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        try:
            _mfl_ver = mfl.__version__
        except AttributeError:
            _mfl_ver = "unknown"
        print(f"mofaflex {_mfl_ver} available | torch {torch.__version__} | device: {DEVICE}")
        if DEVICE == "cpu":
            print("  no CUDA device detected - MOFA-Flex will run, but SVI on CPU is much "
                  "slower than on GPU. Time the pilot cell before committing to the full grid.")
    except Exception as exc:
        print(f"MOFA-Flex disabled ({type(exc).__name__}: {exc}). Requires `pip install mofaflex` "
              f"(Python >= 3.11) plus a torch build matching this machine's CUDA version. The rest "
              f"of this notebook (matrix rebuild, masking) will still run, but every MOFA-Flex "
              f"training cell below will skip with a clear message.")

try:
    THIS_DIR = Path(__file__).resolve().parent
except NameError:
    THIS_DIR = Path.cwd()

BASE_DIR = THIS_DIR.parent  # the project root, e.g. /home/rnjue/Imputationv2

CANDIDATE_PROCESSED_DIRS = []
if os.environ.get("IMPUTATION_OUTPUT_DIR"):
    CANDIDATE_PROCESSED_DIRS.append(Path(os.environ["IMPUTATION_OUTPUT_DIR"]).expanduser())
CANDIDATE_PROCESSED_DIRS += [
    BASE_DIR / "data" / "processed",
    Path.home() / "reuben_imputation" / "processed",
    Path("/projects/datasets_BIO/reuben_imputation/processed"),
    THIS_DIR / "processed",
]
PROCESSED_DIR = next((d for d in CANDIDATE_PROCESSED_DIRS if d.exists()), None)
if PROCESSED_DIR is None:
    raise FileNotFoundError(
        "no 'processed' directory found among: " + ", ".join(str(d) for d in CANDIDATE_PROCESSED_DIRS) +
        ". Run read_multiomic_Data_clinical.ipynb through Cell 14 first, or set IMPUTATION_OUTPUT_DIR."
    )

CANDIDATE_RESULTS_DIRS = [
    BASE_DIR / "results",
    PROCESSED_DIR.parent / "imputation_results",
    Path.home() / "reuben_imputation" / "imputation_results",
]
RESULTS_DIR = next((d for d in CANDIDATE_RESULTS_DIRS if d.exists()), None)
if RESULTS_DIR is None:
    raise FileNotFoundError(
        "no imputation_results directory found among: " + ", ".join(str(d) for d in CANDIDATE_RESULTS_DIRS) +
        ". Run benchmark_imputation_clinical.ipynb first."
    )
# Same shared files evaluate_mofaplus_multiomic_clinical.ipynb writes to - MOFA-Flex's rows
# get appended alongside MOFA+'s and the 25 methods', all in one table.
PILOT_CHECKPOINT_PATH = RESULTS_DIR / "pilot_checkpoint.csv"
PILOT_MATRICES_DIR = RESULTS_DIR / "pilot_imputed_matrices"
PILOT_MATRICES_DIR.mkdir(exist_ok=True)
print("processed dir:", PROCESSED_DIR, "| results dir:", RESULTS_DIR)

mdata = mu.read_h5mu(PROCESSED_DIR / "dlbcl_mudata_filtered.h5mu")
crc_df = pd.read_csv(PROCESSED_DIR / "crc_proteomics_samples_x_features.csv", index_col=0)


def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.asarray(X)


CPU budget for this session: 80 threads (source: CPU affinity (no Slurm allocation, no override detected))
Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.

mofaflex 0.1.dev1+gdf5a17c14 available | torch 2.11.0+cu130 | device: cpu
  no CUDA device detected - MOFA-Flex will run, but SVI on CPU is much slower than on GPU. Time the pilot cell before committing to the full grid.
processed dir: /home/rnjue/Imputationv2/data/processed | results dir: /home/rnjue/Imputationv2/results


In [2]:
# --- Cell 2: rebuild the three per-matrix truth matrices (identical to
# evaluate_mofaplus_multiomic_clinical.ipynb's Cell 2 and
# benchmark_imputation_clinical.ipynb's Cell 3 - same 500-most-variable-feature
# subset, so MOFA-Flex is scored on exactly the same matrices as MOFA+ and the
# 25 conventional methods).
rna_adata = mdata.mod["1_rna"]
assert (rna_adata.obs_names == mdata.obs_names).all()
profiled_mask = mdata.obs["rna_profiled"].to_numpy()

rna_full = pd.DataFrame(
    to_dense(rna_adata.X)[profiled_mask].T, index=rna_adata.var_names, columns=rna_adata.obs_names[profiled_mask]
)
prot_full = pd.DataFrame(to_dense(mdata.mod["2_prot"].X).T, index=mdata.mod["2_prot"].var_names, columns=mdata.obs_names)
crc_full = crc_df.T

N_TOP_FEATURES = 500  # matches benchmark_imputation_clinical.ipynb/py / evaluate_mofaplus_multiomic_clinical.ipynb


def top_variable_features(df, n):
    if df.shape[0] <= n:
        return df
    top_idx = df.var(axis=1, skipna=True).sort_values(ascending=False).index[:n]
    return df.loc[df.index.isin(top_idx)]


MATRICES = {
    "dlbcl_rna": top_variable_features(rna_full, N_TOP_FEATURES),
    "dlbcl_protein": top_variable_features(prot_full, N_TOP_FEATURES),
    "crc_protein": top_variable_features(crc_full, N_TOP_FEATURES),
}
PROTEIN_MATRICES = {"dlbcl_protein", "crc_protein"}
for name, df in MATRICES.items():
    print(f"{name}: {df.shape}")


dlbcl_rna: (500, 318)
dlbcl_protein: (500, 332)
crc_protein: (500, 997)


In [3]:
# --- Cell 3: masking functions and PILOT_ROW (same as what is in
# evaluate_mofaplus_multiomic_clinical.ipynb's Cell 4 - same masking design,
# so MOFA-Flex's scenarios are byte-identical to MOFA+'s and the 25 methods').
def generate_mnar_weighted_positions(X, n_to_mask, slope=-1.5, rng=None, exclude_mask=None):
    if rng is None:
        rng = np.random.default_rng()
    eligible = X.notna().values
    if exclude_mask is not None:
        eligible = eligible & ~exclude_mask.values
    positions = np.argwhere(eligible)
    if n_to_mask == 0:
        return np.empty((0, 2), dtype=int)
    values = X.values[eligible]
    mean_v, std_v = values.mean(), values.std()
    z = np.zeros_like(values) if std_v == 0 else (values - mean_v) / std_v
    weights = np.exp(slope * z)
    weights = weights / weights.sum()
    selected = rng.choice(len(positions), size=n_to_mask, replace=False, p=weights)
    return positions[selected]


def generate_mixed_mask(X, masking_fraction=0.20, mnar_fraction=0.50, slope=-1.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    eligible = X.notna().values
    n_eligible = eligible.sum()
    total_to_mask = int(round(masking_fraction * n_eligible))
    n_mnar = int(round(total_to_mask * mnar_fraction))
    n_mcar = total_to_mask - n_mnar

    mnar_positions = generate_mnar_weighted_positions(X, n_mnar, slope=slope, rng=rng)
    mnar_excl = np.zeros(X.shape, dtype=bool)
    if len(mnar_positions) > 0:
        mnar_excl[mnar_positions[:, 0], mnar_positions[:, 1]] = True
    mnar_excl_df = pd.DataFrame(mnar_excl, index=X.index, columns=X.columns)

    mcar_positions = np.empty((0, 2), dtype=int)
    if n_mcar > 0:
        available = np.argwhere(eligible & ~mnar_excl_df.values)
        n_mcar_clamped = min(n_mcar, len(available))
        mcar_positions = available[rng.choice(len(available), size=n_mcar_clamped, replace=False)]

    mask = np.zeros(X.shape, dtype=bool)
    mechanism = np.full(X.shape, "", dtype=object)
    if len(mnar_positions) > 0:
        mask[mnar_positions[:, 0], mnar_positions[:, 1]] = True
        mechanism[mnar_positions[:, 0], mnar_positions[:, 1]] = "MNAR"
    if len(mcar_positions) > 0:
        mask[mcar_positions[:, 0], mcar_positions[:, 1]] = True
        mechanism[mcar_positions[:, 0], mcar_positions[:, 1]] = "MCAR"
    return (
        pd.DataFrame(mask, index=X.index, columns=X.columns),
        pd.DataFrame(mechanism, index=X.index, columns=X.columns),
    )


def build_masked_scenario(row, X_truth, slope=-1.5):
    rng = np.random.default_rng(int(row["Seed"]))
    mask, mechanism = generate_mixed_mask(
        X_truth, masking_fraction=float(row["Masking_Fraction"]), mnar_fraction=float(row["MNAR_Fraction"]),
        slope=slope, rng=rng,
    )
    return X_truth.mask(mask), mask, mechanism


PILOT_ROW = pd.Series({"Masking_Fraction": 0.10, "MNAR_Fraction": 0.0, "Replicate": 1, "Seed": RNG_SEED})


In [4]:
# --- Cell 4: metrics -- calculate_metrics, pooled_correlations, score_full
# (verbatim from evaluate_mofaplus_multiomic_clinical.ipynb's Cell 5, so
# MOFA-Flex's metrics are computed identically to MOFA+'s and the 25 methods').
from scipy.stats import spearmanr


def calculate_metrics(truth, prediction, mask):
    y_true_mat, y_pred_mat, mask_mat = truth.values, prediction.values, mask.values
    y_true, y_pred = y_true_mat[mask_mat], y_pred_mat[mask_mat]
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[valid], y_pred[valid]

    errors = y_true - y_pred
    rmse = np.sqrt(np.mean(errors ** 2)) if len(errors) else np.nan
    mae = np.mean(np.abs(errors)) if len(errors) else np.nan
    pearson_r = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 1 else np.nan

    nrmse_vals = []
    min_masked_per_feature = 5
    for i in range(y_true_mat.shape[0]):
        row_mask = mask_mat[i]
        if row_mask.sum() < min_masked_per_feature:
            continue
        yt, yp = y_true_mat[i, row_mask], y_pred_mat[i, row_mask]
        v = np.isfinite(yt) & np.isfinite(yp)
        yt, yp = yt[v], yp[v]
        if len(yt) < min_masked_per_feature:
            continue
        feat_range = np.nanmax(y_true_mat[i]) - np.nanmin(y_true_mat[i])
        if not np.isfinite(feat_range) or feat_range <= 0:
            continue
        row_rmse = np.sqrt(np.mean((yt - yp) ** 2))
        nrmse_vals.append(row_rmse / feat_range)

    return {
        "RMSE": rmse, "MAE": mae, "Pearson_r": pearson_r,
        "NRMSE": float(np.mean(nrmse_vals)) if nrmse_vals else np.nan,
        "N_Evaluated": len(y_true),
    }


def pooled_correlations(X_truth, X_hat, mask, min_masked_per_feature=5):
    yt_mat, yp_mat, mask_mat = X_truth.values, X_hat.values, mask.values
    raw_true, raw_pred, cen_true, cen_pred = [], [], [], []
    n_features_used = 0
    for i in range(yt_mat.shape[0]):
        row_mask = mask_mat[i]
        if row_mask.sum() < min_masked_per_feature:
            continue
        yt, yp = yt_mat[i, row_mask], yp_mat[i, row_mask]
        v = np.isfinite(yt) & np.isfinite(yp)
        yt, yp = yt[v], yp[v]
        if len(yt) < min_masked_per_feature:
            continue
        raw_true.append(yt)
        raw_pred.append(yp)
        cen_true.append(yt - yt.mean())
        cen_pred.append(yp - yp.mean())
        n_features_used += 1

    if n_features_used == 0:
        return {"Pearson_r_raw_pooled": np.nan, "Spearman_r_raw_pooled": np.nan,
                "Pearson_r_feature_centered_pooled": np.nan, "Spearman_r_feature_centered_pooled": np.nan,
                "N_Features_Pooled": 0}

    raw_true, raw_pred = np.concatenate(raw_true), np.concatenate(raw_pred)
    cen_true, cen_pred = np.concatenate(cen_true), np.concatenate(cen_pred)

    def _pearson(a, b):
        return np.corrcoef(a, b)[0, 1] if len(a) > 1 and np.std(a) > 0 and np.std(b) > 0 else np.nan

    def _spearman(a, b):
        return spearmanr(a, b).correlation if len(a) > 1 and np.std(a) > 0 and np.std(b) > 0 else np.nan

    return {
        "Pearson_r_raw_pooled": _pearson(raw_true, raw_pred),
        "Spearman_r_raw_pooled": _spearman(raw_true, raw_pred),
        "Pearson_r_feature_centered_pooled": _pearson(cen_true, cen_pred),
        "Spearman_r_feature_centered_pooled": _spearman(cen_true, cen_pred),
        "N_Features_Pooled": n_features_used,
    }


def score_full(matrix_name, method_name, X_truth, X_hat, mask, t0, scenario_row=None):
    """Bundles calculate_metrics + pooled_correlations into one row, in the
    same shape pilot_checkpoint.csv expects. scenario_row defaults to
    PILOT_ROW (pilot usage); full-grid usage passes the scenario's own row."""
    row = PILOT_ROW if scenario_row is None else scenario_row
    X_hat = X_hat.reindex(index=X_truth.index, columns=X_truth.columns)
    m = calculate_metrics(X_truth, X_hat, mask)
    m.update(pooled_correlations(X_truth, X_hat, mask))
    m.update({"Matrix": matrix_name, "Method": method_name, "Seconds": time.time() - t0,
              **{c: row[c] for c in ["Masking_Fraction", "MNAR_Fraction", "Replicate", "Seed"]}})
    return m, X_hat


In [5]:
# --- Cell 5: mofa_flex_impute() --- trains MOFA-Flex and reconstructs the
# completed matrix. 
import anndata as ad
import contextlib, io as _io

N_FACTORS = 15  # matches evaluate_mofaplus_multiomic_clinical.ipynb's N_FACTORS, for direct comparability


def mofa_flex_impute(Y_masked, n_factors=N_FACTORS, seed=RNG_SEED, max_epochs=2000,
                      early_stopper_patience=200, n_particles=5, device=None):
    """MOFA-Flex probabilistic matrix completion via Pyro-backed variational inference.

    Single-view, single-group model with factor_prior="Normal" and
    weight_prior="SpikeSlab" - matches MOFA+'s default ARD-sparsity configuration
    (see this notebook's header markdown). Missing entries (both the matrix's own
    real missingness and the synthetic mask) are handled natively - excluded from
    the likelihood during training, not pre-filled.

    Parameters
    ----------
    Y_masked : pandas.DataFrame
        features x samples, NaN = missing (real or masked). Works for dlbcl_rna,
        dlbcl_protein, or crc_protein identically - no matrix-specific logic here.

    Returns
    -------
    tuple[pandas.DataFrame, dict]
        Completed features x samples matrix, and a diagnostics dict (training loss
        trace, epochs run, per-factor R2 if available).

    Notes
    -----
    n_particles=5 and early_stopper_patience=200 (over mofaflex's own defaults of
    1 and 100) -  single-particle ELBO estimates are noisy enough to trigger premature 
    early stopping on this kind of data; multi-particle sampling stabilizes the gradient instead.
    """
    if device is None:
        device = DEVICE

    Yc = Y_masked.T  # samples x features -- mofaflex/AnnData convention

    # --- fully-missing-feature guard ---------------------------------------------
    # mofaflex's 'random' init computes a per-feature statistic from that feature's
    # observed entries; with zero observed entries the statistic is NaN, which
    # propagates into the likelihood's `loc` parameter and crashes at initialization
    # with "Expected parameter loc ... to satisfy the constraint Real(), but found
    # invalid values." At the highest masking/MNAR corners of the grid, a matrix's
    # real baseline missingness plus the synthetic mask can push a feature into 100%
    # missing. This mirrors the identical fully-missing-row failure
    # mode already fixed for bpca_impute()/dreamai_impute() in
    # benchmark_imputation_clinical.ipynb (Cell 12) - seed the affected feature(s)
    # with a neutral constant (the grand mean) so mofaflex's init statistics stay
    # finite, then blank the corresponding output rows back to NaN afterward (see
    # below) so a feature the model was never given real signal about is never
    # scored as a genuine prediction.
    fully_missing_features = Yc.isna().all(axis=0)
    if fully_missing_features.any():
        grand_mean = float(np.nanmean(Yc.values))
        if not np.isfinite(grand_mean):
            grand_mean = 0.0
        Yc = Yc.copy()
        Yc.loc[:, fully_missing_features] = grand_mean

    feature_shift = Yc.mean(axis=0, skipna=True)
    centered = (Yc - feature_shift).values
    global_scale = float(np.sqrt(np.nanvar(centered)))

    adata = ad.AnnData(X=Yc.values.astype(np.float32))
    adata.obs_names = Yc.index.astype(str)
    adata.var_names = Yc.columns.astype(str)
    data_dict = {"group1": {"features": adata}}

    with contextlib.redirect_stdout(_io.StringIO()):
        model = mfl.terms.MofaFlex(n_factors=n_factors, factor_prior="Normal", weight_prior="SpikeSlab")
        model.fit(
            data_dict,
            likelihoods="Normal",
            device=device,
            max_epochs=max_epochs,
            early_stopper_patience=early_stopper_patience,
            n_particles=n_particles,
            seed=seed,
            batch_size=0,                    # whole matrix as one batch - 500 x (samples) is small
            plot_data_overview=False,
            subset_var=None,
            remove_constant_features=False,  # keep every feature for shape-consistent scoring
            update_every=-1,
            save_path=False,
        )
        imputed = model.impute_data(data_dict, missing_only=False)

    Xhat_preprocessed = pd.DataFrame(
        np.asarray(imputed["group1"]["features"].X), index=Yc.index, columns=Yc.columns
    )
    Xhat = (Xhat_preprocessed * global_scale + feature_shift).T  # back to features x samples, original scale

    if fully_missing_features.any():
        # blank these features' predictions back to NaN - the model never saw a
        # real observation for them, so whatever it output is not a genuine
        # prediction and must not be scored as one (mirrors
        # _blank_out_unidentifiable_rows()'s role for bpca_impute()/dreamai_impute()).
        Xhat.loc[fully_missing_features[fully_missing_features].index, :] = np.nan

    diagnostics = {"model": model, "feature_shift": feature_shift, "global_scale": global_scale,
                    "n_fully_missing_features_blanked": int(fully_missing_features.sum())}
    diagnostics["training_loss"] = np.asarray(model.training_loss)
    diagnostics["n_epochs_run"] = int(len(diagnostics["training_loss"]))
    diagnostics["max_epochs"] = max_epochs
    diagnostics["early_stopped"] = diagnostics["n_epochs_run"] < max_epochs
    try:
        diagnostics["r2"] = model.get_r2()
    except Exception as exc:
        diagnostics["r2"] = None
        diagnostics["r2_error"] = f"{type(exc).__name__}: {exc}"

    return Xhat, diagnostics


In [6]:
# --- Cell 6: pilot scenario, per-matrix baseline ("MOFA-Flex") ------------
# Checkpointed into the same pilot_checkpoint.csv / pilot_imputed_matrices/
# that evaluate_mofaplus_multiomic_clinical.ipynb and the 25-method benchmark
# use, so MOFA-Flex's rows land alongside theirs (Method="MOFA-Flex").
if PILOT_CHECKPOINT_PATH.exists():
    _checkpoint_df = pd.read_csv(PILOT_CHECKPOINT_PATH)
    pilot_rows = _checkpoint_df.to_dict("records")
else:
    pilot_rows = []
_pilot_row_index = {(r["Matrix"], r["Method"]): i for i, r in enumerate(pilot_rows)}


def _pilot_matrix_path(matrix_name, method_name):
    return PILOT_MATRICES_DIR / f"pilot_{matrix_name}_imputed_{method_name}.csv"


def _record_pilot_result(matrix_name, method_name, m, X_hat):
    matrix_path = _pilot_matrix_path(matrix_name, method_name)
    X_hat.rename_axis("Feature_ID").to_csv(matrix_path)
    if (matrix_name, method_name) in _pilot_row_index:
        pilot_rows[_pilot_row_index[(matrix_name, method_name)]] = m
    else:
        pilot_rows.append(m)
        _pilot_row_index[(matrix_name, method_name)] = len(pilot_rows) - 1
    pd.DataFrame(pilot_rows).to_csv(PILOT_CHECKPOINT_PATH, index=False)


METHOD_NAME = "MOFA-Flex"

if not MOFAFLEX_AVAILABLE:
    print("skipping MOFA-Flex pilot - mofaflex/torch not available")
else:
    for matrix_name, X in MATRICES.items():
        matrix_path = _pilot_matrix_path(matrix_name, METHOD_NAME)
        if (matrix_name, METHOD_NAME) in _pilot_row_index and matrix_path.exists():
            print(f"{matrix_name}/{METHOD_NAME}: skipped - checkpoint + saved matrix both present")
            continue
        print(f"\n=== {METHOD_NAME}: {matrix_name} {X.shape} ===")
        Y_masked, mask, mechanism = build_masked_scenario(PILOT_ROW, X)
        t0 = time.time()
        try:
            X_hat_raw, diag = mofa_flex_impute(Y_masked, seed=RNG_SEED)
            m, X_hat = score_full(matrix_name, METHOD_NAME, X, X_hat_raw, mask, t0)
            n_epochs_run = diag["n_epochs_run"]
            max_epochs_run = diag["max_epochs"]
            stop_reason = "early-stopped" if diag["early_stopped"] else "hit max_epochs"
            print(f"  {n_epochs_run}/{max_epochs_run} epochs run ({stop_reason})")
        except Exception as exc:
            m = {"Matrix": matrix_name, "Method": METHOD_NAME, "Seconds": time.time() - t0,
                 "RMSE": np.nan, "Error": f"{type(exc).__name__}: {exc}"}
            X_hat = None
            print(f"  !! {METHOD_NAME} failed on {matrix_name}: {exc}")
        if X_hat is not None:
            _record_pilot_result(matrix_name, METHOD_NAME, m, X_hat)
        rmse_val = m.get("RMSE", float("nan"))
        print(f"  RMSE={rmse_val:.4f}  ({m['Seconds']:.2f}s)")
        print(f" - multiply by however many full-grid scenarios you plan to run (Cell 7) for a "
              f"rough time estimate before committing to it.")


dlbcl_rna/MOFA-Flex: skipped - checkpoint + saved matrix both present
dlbcl_protein/MOFA-Flex: skipped - checkpoint + saved matrix both present
crc_protein/MOFA-Flex: skipped - checkpoint + saved matrix both present


In [7]:
# --- Cell 7: full-grid MOFA-Flex, checkpointed into benchmark_checkpoint_<matrix>.csv ----
# Mirrors evaluate_mofaplus_multiomic_clinical.ipynb's own full-grid cell
# (run_mofaplus_full_grid / MOFAPLUS_FULL_GRID_ALL_REPLICATES) as closely as possible -
# same MASKING_FRACTIONS/MNAR_FRACTIONS/N_REPLICATES, same Seed formula
# (RNG_SEED * 10_000 + Scenario_ID), same SLOW-tier scaffolding (a no-op with the
# current single-value MASKING_FRACTIONS grid, kept only so this cell doesn't silently
# diverge if the grid is widened again later) - so MOFA-Flex's Scenario_ID/Seed line
# up exactly with MOFA+'s, the 25 conventional methods', and (once built)
# MOFA-Flex-GraphGP's, and all of their rows merge cleanly in
# evaluate_imputation_methods_clinical.ipynb.
#
# Checkpointed into the SAME benchmark_checkpoint_<matrix>.csv files that
# benchmark_imputation_clinical.ipynb/py's full-grid loop and MOFA+'s own full-grid
# cell write to - MOFA-Flex's rows land alongside theirs, no separate table. Don't run
# this cell at the same time as those other full-grid cells for the same matrix (both
# do a read-modify-write against the same checkpoint file); run them one after another
# instead - each resumes from whatever is already checkpointed.
#
# SVI training is not free: budget real wall-clock time before running this on all
# three matrices with MOFAFLEX_FULL_GRID_ALL_REPLICATES=True (5x the scenario count of
# Replicate == 1 alone). Time one matrix first.

MASKING_FRACTIONS = [0.30]
MNAR_FRACTIONS = [0.00, 0.25, 0.50, 0.75, 1.00]
N_REPLICATES = 5  # -> 25 scenarios total, same grid as benchmark_imputation_clinical.ipynb/py,
                  # evaluate_mofaplus_multiomic_clinical.ipynb, and evaluate_imputation_methods_clinical.ipynb

SLOW_GRID_MASKING = {0.30}  # matches benchmark_imputation_clinical.ipynb/py's SLOW_NAMES tier
SLOW_GRID_MNAR = {0.00, 0.25, 0.50, 0.75, 1.00}

# Mirrors MOFA+_native's MOFAPLUS_FULL_GRID_ALL_REPLICATES toggle: True runs MOFA-Flex on
# every replicate, not just Replicate == 1. Set to False to fall back to the
# Replicate == 1 -only subset (5 scenarios/matrix) if the full 25/matrix is too slow.
MOFAFLEX_FULL_GRID_ALL_REPLICATES = True

_rows = []
_scenario_id = 0
for mf in MASKING_FRACTIONS:
    for nf in MNAR_FRACTIONS:
        for rep in range(1, N_REPLICATES + 1):
            _scenario_id += 1
            _rows.append({"Scenario_ID": _scenario_id, "Masking_Fraction": mf, "MNAR_Fraction": nf,
                          "Replicate": rep, "Seed": RNG_SEED * 10_000 + _scenario_id})
experiment_design = pd.DataFrame(_rows)

_replicate_ok = (
    experiment_design["Replicate"] >= 1  # no-op: every replicate qualifies
    if MOFAFLEX_FULL_GRID_ALL_REPLICATES
    else experiment_design["Replicate"] == 1
)
mofaflex_full_grid_design = experiment_design[
    _replicate_ok
    & (experiment_design["Masking_Fraction"].isin(SLOW_GRID_MASKING))
    & (experiment_design["MNAR_Fraction"].isin(SLOW_GRID_MNAR))
].reset_index(drop=True)
print(
    f"MOFA-Flex full-grid scenario subset: {len(mofaflex_full_grid_design)} of {len(experiment_design)} "
    f"scenarios x {len(MATRICES)} matrices = {len(mofaflex_full_grid_design) * len(MATRICES)} total training runs"
)

FULL_GRID_METHOD_NAME = "MOFA-Flex"


def run_mofaflex_full_grid(matrix_name, X, checkpoint_path):
    """Same skip/resume/checkpoint-after-every-scenario contract as
    run_mofaplus_full_grid / benchmark_imputation_clinical.ipynb/py's run_full_benchmark(),
    restricted to mofaflex_full_grid_design and a single method."""
    if checkpoint_path.exists():
        all_results = pd.read_csv(checkpoint_path).to_dict("records")
        print(f"{matrix_name}: resuming from checkpoint, {len(all_results)} result(s) already in file.")
    else:
        all_results = []
    # Only rows WITHOUT a recorded Error count as "done" - a scenario that crashed on
    # a previous run (e.g. the fully-missing-feature NaN-init crash)
    # is retried automatically on resume instead of being silently skipped forever.
    already_done = {(r["Scenario_ID"], r["Method"]) for r in all_results if pd.isna(r.get("Error"))}
    n_failed_retry = sum(
        1 for r in all_results
        if pd.notna(r.get("Error")) and r["Method"] == FULL_GRID_METHOD_NAME
        and (r["Scenario_ID"], r["Method"]) not in already_done
    )
    if n_failed_retry:
        print(f"{matrix_name}: {n_failed_retry} previously-failed {FULL_GRID_METHOD_NAME} "
              f"scenario(s) will be retried this run.")

    for _, row in mofaflex_full_grid_design.iterrows():
        if (row["Scenario_ID"], FULL_GRID_METHOD_NAME) in already_done:
            continue
        Y_masked, mask, mechanism = build_masked_scenario(row, X)
        mcar_only = pd.DataFrame(mechanism.values == "MCAR", index=mask.index, columns=mask.columns)
        mnar_only = pd.DataFrame(mechanism.values == "MNAR", index=mask.index, columns=mask.columns)
        t0 = time.time()
        try:
            # seed=row["Seed"], not the module-level RNG_SEED default - each scenario
            # gets its own SVI training initialization, tied to its own scenario the same
            # way build_masked_scenario above already derives its masking rng from
            # row["Seed"]. Without this every scenario would train from the same fixed
            # init, which is fine for the single-scenario pilot cell (Cell 6) but wrong
            # here, where up to 25 distinct scenarios run back to back.
            X_hat_raw, diag = mofa_flex_impute(Y_masked, seed=int(row["Seed"]))
            X_hat = X_hat_raw.reindex(index=X.index, columns=X.columns)
            overall = calculate_metrics(X, X_hat, mask)
            rec = {"Scenario_ID": row["Scenario_ID"], "Method": FULL_GRID_METHOD_NAME,
                   "Masking_Fraction": row["Masking_Fraction"], "MNAR_Fraction": row["MNAR_Fraction"],
                   "Replicate": row["Replicate"], "Seconds": time.time() - t0, **overall}
            if mcar_only.values.any():
                rec["RMSE_MCAR_subset"] = calculate_metrics(X, X_hat, mcar_only)["RMSE"]
            if mnar_only.values.any():
                rec["RMSE_MNAR_subset"] = calculate_metrics(X, X_hat, mnar_only)["RMSE"]
        except Exception as exc:
            rec = {"Scenario_ID": row["Scenario_ID"], "Method": FULL_GRID_METHOD_NAME,
                   "Masking_Fraction": row["Masking_Fraction"], "MNAR_Fraction": row["MNAR_Fraction"],
                   "Replicate": row["Replicate"], "Seconds": time.time() - t0, "RMSE": np.nan,
                   "Error": f"{type(exc).__name__}: {exc}"}
            print(f"    !! {matrix_name}/{FULL_GRID_METHOD_NAME} failed on scenario {row['Scenario_ID']}: {exc}")
        all_results.append(rec)
        already_done.add((row["Scenario_ID"], FULL_GRID_METHOD_NAME))
        pd.DataFrame(all_results).to_csv(checkpoint_path, index=False)  # checkpoint after each scenario
        rmse_disp = rec.get("RMSE", float("nan"))
        print(
            f"  scenario {int(row['Scenario_ID']):3d} (mask={row['Masking_Fraction']:.2f}, "
            f"mnar={row['MNAR_Fraction']:.2f}): RMSE={rmse_disp:.4f} ({rec['Seconds']:.1f}s)"
        )

    return pd.DataFrame(all_results)


# Run one matrix at a time (if you don't want to run full-grid just comment out) 
# - each call resumes from its own checkpoint if interrupted, so it's safe to run 
# matrices in separate sessions. Time the first matrix before committing to all three.
mofaflex_full_grid_results = {}
if MOFAFLEX_AVAILABLE:
     for matrix_name, X in MATRICES.items():
         print(f"\n=== full-grid MOFA-Flex: {matrix_name} {X.shape} ===")
         mofaflex_full_grid_results[matrix_name] = run_mofaflex_full_grid(
             matrix_name, X, RESULTS_DIR / f"benchmark_checkpoint_{matrix_name}.csv"
         )


MOFA-Flex full-grid scenario subset: 25 of 25 scenarios x 3 matrices = 75 total training runs

=== full-grid MOFA-Flex: dlbcl_rna (500, 318) ===
dlbcl_rna: resuming from checkpoint, 475 result(s) already in file.

=== full-grid MOFA-Flex: dlbcl_protein (500, 332) ===
dlbcl_protein: resuming from checkpoint, 739 result(s) already in file.
dlbcl_protein: 3 previously-failed MOFA-Flex scenario(s) will be retried this run.


WARNING	Could not import dask. Data arrays may be copied, resulting in high memory usage.
INFO	Initializing factors using 'random' method...


INFO	Initializing factors using 'random' method...


  scenario  23 (mask=0.30, mnar=1.00): RMSE=0.6558 (65.9s)


INFO	Initializing factors using 'random' method...


  scenario  24 (mask=0.30, mnar=1.00): RMSE=0.6449 (63.1s)
  scenario  25 (mask=0.30, mnar=1.00): RMSE=0.6407 (63.3s)

=== full-grid MOFA-Flex: crc_protein (500, 997) ===
crc_protein: resuming from checkpoint, 700 result(s) already in file.
